In [1]:
#Function taken from Assignment 1

import subprocess
import sys
import os

def run_srilm(command, warnings=False):
    """Runs the provided SRILM command and prints any resulting messages
    
    Arguments
    ---------
    command: str; the SRILM command, as it would be entered in the Terminal
    warnings: bool; whether or not to print warnings alongside output
    """
    if sys.platform == 'win32':
        # This is just so that it works on my Windows machine.
        # For the purposes of this class, just ignore this line!
        command = [r"C:\cygwin64\bin\bash.exe ", "--login", "-c",
            "source ~/.bashrc && cd / && /srilm/bin/cygwin64/" + command]

    stdout = subprocess.PIPE
    if warnings:
        stderr = subprocess.STDOUT
    else:
        stderr = subprocess.PIPE
        
    try:
        result = subprocess.run(command, shell=True, check=True, stdout=stdout, stderr=stderr)
        output_msg = result.stdout
        if output_msg:
            print(output_msg.decode('UTF-8'))
        else:
            print("The command ran successfully")
            
    except subprocess.CalledProcessError as e:
        print("The command did not run, and produced the following error message:\n{}".format(e.stderr.decode('UTF-8')))

In [2]:
#Import the SRILM-ready data set made in the preprocessing nb
from pathlib import Path

file_path = Path("../output/countries/")
txt_files = sorted(file_path.glob("*.txt"))

print(f"Imported {len(txt_files)} files:")
print([f.name for f in txt_files])

Imported 9 files:
['CZ.txt', 'DE.txt', 'FR.txt', 'GR.txt', 'NL.txt', 'PL.txt', 'PO.txt', 'SP.txt', 'UK.txt']


In [19]:
#split into train and test files using 80/20 split
import random

input_directory = Path("../output/countries")
train_directory = Path("../output/train")
test_directory = Path("../output/test")

train_directory.mkdir(parents=True, exist_ok=True)
test_directory.mkdir(parents=True, exist_ok=True)

random.seed(2026)
start = 0
skipped = []


for file_path in input_directory.glob("*.txt"):
    nation = file_path.stem
    
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    #shuffle the speeches so there is no issues with the time of the speech impacting results
    random.shuffle(lines)

    if len(lines) > 2000:
        
        
        split_idx = int(len(lines) * 0.8)
        train_lines = lines[:split_idx]
        test_lines = lines[split_idx:]
    
        #making the training sets
        train_file = train_directory / f"{nation}_train.txt" 
        with open(train_file, "w", encoding="utf-8") as f:
            f.write("\n".join(train_lines))
    
        #making the test sets
        test_file = test_directory / f"{nation}_test.txt"
        with open(test_file, "w", encoding="utf-8") as f:
            f.write("\n".join(test_lines))
    
    
        if start == 0:
            print("The splits are:")
            print("-" * 30)
            start += 1
        print(f"{nation}: {len(train_lines)} train / {len(test_lines)} test")

    else:
        skipped.append(nation)
        pass
print("-" * 20)
print(f"Nations skipped are: {skipped}")

The splits are:
------------------------------
DE: 3948 train / 987 test
GR: 132322 train / 33081 test
UK: 95357 train / 23840 test
PL: 12380 train / 3095 test
SP: 7242 train / 1811 test
NL: 33656 train / 8414 test
--------------------
Nations skipped are: ['CZ', 'FR', 'PO']


In [4]:
#make the models folder for SRILM to input made models into

model_directory = Path("../output/models")
model_directory.mkdir(parents=True, exist_ok=True)



In [5]:
#bigrams

for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"Trigram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -text ../output/train/{name}_train.txt -order 2 -lm ../output/models/{name}_trigram.lm -kndiscount")
        run_srilm(f"ngram -order 2 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_trigram.lm")

Trigram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30354 words, 749 OOVs
0 zeroprobs, logprob= -66499.35 ppl= 149.1934 ppl1= 176.287

Trigram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 33081 sentences, 635733 words, 2333 OOVs
0 zeroprobs, logprob= -1459930 ppl= 155.0618 ppl1= 201.7946

Trigram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 251352 words, 218 OOVs
0 zeroprobs, logprob= -517759.8 ppl= 98.82156 ppl1= 115.2623

Trigram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 53989 words, 1348 OOVs
0 zeroprobs, logprob= -129494.4 ppl= 210.5488 ppl1= 288.372

Trigram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 44436 words, 1086 OOVs
0 zeroprobs, logprob

In [6]:
#Trigrams
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"Trigram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -text ../output/train/{name}_train.txt -order 3 -lm ../output/models/{name}_trigram.lm -kndiscount")
        run_srilm(f"ngram -order 3 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_trigram.lm")


Trigram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30354 words, 749 OOVs
0 zeroprobs, logprob= -65063.42 ppl= 133.9095 ppl1= 157.6584

Trigram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 33081 sentences, 635733 words, 2333 OOVs
0 zeroprobs, logprob= -1390580 ppl= 122.0257 ppl1= 156.8274

Trigram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 251352 words, 218 OOVs
0 zeroprobs, logprob= -379944.4 ppl= 29.09842 ppl1= 32.57729

Trigram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 53989 words, 1348 OOVs
0 zeroprobs, logprob= -126693.2 ppl= 187.5406 ppl1= 255.1178

Trigram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 44436 words, 1086 OOVs
0 zeroprobs, logpr

In [7]:
#fourgram model (WINNER)

print("KND UNINT")
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"4-gram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram.lm -kndiscount")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram.lm")


KND UNINT
4-gram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30354 words, 0 OOVs
0 zeroprobs, logprob= -66592.15 ppl= 133.2789 ppl1= 156.2611

4-gram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 33081 sentences, 635733 words, 0 OOVs
0 zeroprobs, logprob= -1389611 ppl= 119.598 ppl1= 153.4054

4-gram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 251352 words, 0 OOVs
0 zeroprobs, logprob= -300617.7 ppl= 14.36361 ppl1= 15.70373

4-gram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 53989 words, 0 OOVs
0 zeroprobs, logprob= -129256.6 ppl= 183.7905 ppl1= 247.8157

4-gram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 44436 words, 0 OOVs
0 zeroprobs, logprob= -9775

In [8]:
#fivegram 
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"5-gram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -text ../output/train/{name}_train.txt -order 5 -lm ../output/models/{name}_fivegram.lm -kndiscount")
        run_srilm(f"ngram -order 5 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fivegram.lm")


5-gram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30354 words, 749 OOVs
0 zeroprobs, logprob= -65194.36 ppl= 135.2359 ppl1= 159.2723

5-gram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 33081 sentences, 635733 words, 2333 OOVs
0 zeroprobs, logprob= -1384784 ppl= 119.6066 ppl1= 153.5577

5-gram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 251352 words, 218 OOVs
0 zeroprobs, logprob= -275505.8 ppl= 11.52081 ppl1= 12.50394

5-gram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 53989 words, 1348 OOVs
0 zeroprobs, logprob= -126784.7 ppl= 188.2509 ppl1= 256.141

5-gram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 44436 words, 1086 OOVs
0 zeroprobs, logprob= -9

In [9]:
#Smoothing methods

#KND INT
print("KND INT)")
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"4-gram for {name}")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_INT.lm -kndiscount -interpolate")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_INT.lm")


KND INT)
4-gram for DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30354 words, 0 OOVs
0 zeroprobs, logprob= -67935.76 ppl= 147.1067 ppl1= 173.0277

4-gram for GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 33081 sentences, 635733 words, 0 OOVs
0 zeroprobs, logprob= -1375325 ppl= 113.8582 ppl1= 145.6698

4-gram for NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 251352 words, 0 OOVs
0 zeroprobs, logprob= -276788.8 ppl= 11.62871 ppl1= 12.62409

4-gram for PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 53989 words, 0 OOVs
0 zeroprobs, logprob= -132227.5 ppl= 207.1893 ppl1= 281.2915

4-gram for SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 44436 words, 0 OOVs
0 zeroprobs, logprob= -1000

In [10]:
#Modified KN Discount W/o interpolation


print("Modified KND)")
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"{name}")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_uknd.lm -ukndiscount")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_uknd.lm")


Modified KND)
DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30354 words, 0 OOVs
0 zeroprobs, logprob= -65955.18 ppl= 127.1856 ppl1= 148.8903

GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 33081 sentences, 635733 words, 0 OOVs
0 zeroprobs, logprob= -1380161 ppl= 115.7695 ppl1= 148.2434

NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 251352 words, 0 OOVs
0 zeroprobs, logprob= -218786.5 ppl= 6.954155 ppl1= 7.420591

PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 53989 words, 0 OOVs
0 zeroprobs, logprob= -128227.2 ppl= 176.3153 ppl1= 237.1712

SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 44436 words, 0 OOVs
0 zeroprobs, logprob= -96911.97 ppl= 124.6034 ppl1= 151.6819

UK
------------

In [11]:
#Modified KN Discount W/ interpolation


print("Modified KND with interpolation")
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"{name}")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_uknd_int.lm -ukndiscount -interpolate")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_uknd_int.lm")


Modified KND with interpolation
DE
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30354 words, 0 OOVs
0 zeroprobs, logprob= -68113.65 ppl= 149.0418 ppl1= 175.3784

GR
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 33081 sentences, 635733 words, 0 OOVs
0 zeroprobs, logprob= -1378171 ppl= 114.979 ppl1= 147.1787

NL
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 251352 words, 0 OOVs
0 zeroprobs, logprob= -216558.1 ppl= 6.81814 ppl1= 7.270644

PL
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 53989 words, 0 OOVs
0 zeroprobs, logprob= -132562.5 ppl= 210.0083 ppl1= 285.3396

SP
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 44436 words, 0 OOVs
0 zeroprobs, logprob= -100245.4 ppl= 147.0989 ppl1= 180.2814


In [12]:
#BEST MODELS (per lang)

for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    elif name in ["DE","PL" ,"SP"]:
        print(f"{name} with no interpolation")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_uknd.lm -ukndiscount")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_uknd.lm")

    else:
        print(f"{name} with interpolation")
        print("-" * 30)
        run_srilm(f"ngram-count -unk -text ../output/train/{name}_train.txt -order 4 -lm ../output/models/{name}_fourgram_uknd_int.lm -ukndiscount -interpolate")
        run_srilm(f"ngram -order 4 -unk -ppl ../output/test/{name}_test.txt -lm ../output/models/{name}_fourgram_uknd_int.lm")

DE with no interpolation
------------------------------
The command ran successfully
file ../output/test/DE_test.txt: 987 sentences, 30354 words, 0 OOVs
0 zeroprobs, logprob= -65955.18 ppl= 127.1856 ppl1= 148.8903

GR with interpolation
------------------------------
The command ran successfully
file ../output/test/GR_test.txt: 33081 sentences, 635733 words, 0 OOVs
0 zeroprobs, logprob= -1378171 ppl= 114.979 ppl1= 147.1787

NL with interpolation
------------------------------
The command ran successfully
file ../output/test/NL_test.txt: 8414 sentences, 251352 words, 0 OOVs
0 zeroprobs, logprob= -216558.1 ppl= 6.81814 ppl1= 7.270644

PL with no interpolation
------------------------------
The command ran successfully
file ../output/test/PL_test.txt: 3095 sentences, 53989 words, 0 OOVs
0 zeroprobs, logprob= -128227.2 ppl= 176.3153 ppl1= 237.1712

SP with no interpolation
------------------------------
The command ran successfully
file ../output/test/SP_test.txt: 1811 sentences, 44436 wor

In [17]:
#Testing the models to see their outputs for each country
for file in txt_files:
    name = file.name[:-4]
    if name in skipped:
        pass
    else:
        print(f"10 sample outputs for {name}")
        print("-" * 20)
        run_srilm(f"ngram -lm ../output/models/{name}_fourgram_uknd_int.lm -seed 2026 -gen 10")

10 sample outputs for DE
--------------------
it will continue to support the environmental
i think it will continue to submit to the priorities tax collection and from the use of the only about both received a conduct these let me in terms of member states in the issue of public facts the demographic developments in the future even as an important debate in our country has been successful
we decided needs of the recent developments terezin reminder of destabilizing originate fight their freedom submitted to the state by retirement i had the courage in the development of the eu committee which rejected eventual sixteen able to move terezín new advanced as return policy
it to enable the maximum
on the location a maximum transparency
in a cooperation in western in new citizens in the first point with the others a very important for us to him doputovat sovereignly as on earn law enforcement
this means that will be pension reform parliament as soon we we come to join the need for shaken to